<a href="https://colab.research.google.com/github/EliasNoorzad/temporal-reasoning-TISER/blob/main/notebooks/Extension_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/EliasNoorzad/temporal-reasoning-TISER.git

Cloning into 'temporal-reasoning-TISER'...
remote: Enumerating objects: 330, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 330 (delta 56), reused 88 (delta 28), pack-reused 207 (from 1)
Receiving objects: 100% (330/330), 470.00 KiB | 21.36 MiB/s, done.
Resolving deltas: 100% (185/185), done.


In [2]:
%cd temporal-reasoning-TISER
!ls

/content/temporal-reasoning-TISER
checkpoints  extensions  README.md	   results
configs      notebooks	 requirements.txt  src


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.5 MB/s eta 0:00:00


In [6]:
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi

HF_REPO = "EliElias/TISER-Evaluation-Results"

INPUT_PATH = hf_hub_download(
    repo_id=HF_REPO,
    filename="lora_both_results_rescored.jsonl",
    repo_type="dataset",
)

OUTPUT_DIR = Path("/content/extension_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

lora_both_results_rescored.jsonl: reconstructing file:   0%|          |  0.00B / 47.1MB            

lora_both_results_rescored.jsonl: downloading bytes:           |  0.00B            

In [7]:
!python extensions/adaptive_routing/tfidf_analysis.py \
    --input "{INPUT_PATH}" \
    --output "{OUTPUT_DIR}/tfidf_analysis.jsonl" \
    --summary-output "{OUTPUT_DIR}/tfidf_analysis_summary.csv"

Loading examples: 20442records [00:00, 93631.30records/s]
Preparing TF-IDF corpus: 100% 20442/20442 [00:00<00:00, 42465.18examples/s]
Computing TF-IDF features: 100% 20442/20442 [00:20<00:00, 981.01examples/s]
Writing TF-IDF features: 100% 20442/20442 [00:00<00:00, 50751.48records/s]
Loaded examples: 20442
Total TF-IDF documents: 224129
TF-IDF vocabulary size: 47102
Output path: /content/extension_1/tfidf_analysis.jsonl
No-overlap examples: 0 (0.00%)


In [8]:
!python extensions/adaptive_routing/tfidf_validation.py \
    --input "{OUTPUT_DIR}/tfidf_analysis.jsonl" \
    --output-dir "{OUTPUT_DIR}"

Loaded examples: 19102

TF-IDF concentration quartiles
    quartile  examples  raw_feature_mean  raw_feature_median  direct_em  tiser_em  em_gain  direct_f1  tiser_f1  f1_gain  both_correct_rate  tiser_rescue_rate
 Q1 - Lowest      4776            0.4289              0.4027    86.6206   90.3476   3.7270    90.2417   93.1379   2.8962            83.8358             6.5117
          Q2      4776            0.2846              0.2782    85.6575   90.3476   4.6901    90.1506   93.4184   3.2678            83.2705             7.0771
          Q3      4774            0.1939              0.1927    78.1734   86.3217   8.1483    83.8354   90.3313   6.4960            75.8065            10.5153
Q4 - Highest      4776            0.1140              0.1183    68.9280   79.0829  10.1549    78.9576   86.2438   7.2862            65.4104            13.6725

TF-IDF concentration correlations
Direct EM: rho=-0.167435
Direct F1: rho=-0.161802
TISER EM gain: rho=0.076910
TISER F1 gain: rho=0.089606

TF-IDF c

In [9]:
!python extensions/adaptive_routing/tfidf_router.py \
    --input "{OUTPUT_DIR}/tfidf_analysis.jsonl" \
    --output "{OUTPUT_DIR}/tfidf_router_sweep.csv"

Loaded examples: 19102

Always Direct:
  Macro EM: 79.07%
  Macro F1: 85.54%
  Average generated tokens: 5.93
  Total generated tokens: 113257

Always TISER:
  Macro EM: 86.14%
  Macro F1: 90.73%
  Average generated tokens: 272.00
  Total generated tokens: 5195747

Both-correct examples: 14724
TISER-rescue examples: 1804

Threshold sweep (EM/F1 columns are five-dataset macro percentages)
 threshold  routed_em  routed_f1  avg_generated_tokens  total_generated_tokens  direct_count  direct_pct  tiser_count  tiser_pct  token_saving_vs_tiser_pct  both_correct_to_direct  both_correct_coverage_pct  rescued_sent_to_direct  rescue_loss_pct  rescue_preserved_pct  tiser_gain_retained_pct
         3    85.6877    90.4716              223.8255            4275515.0000          3903     20.4324        15199    79.5676                    17.7113                    3372                    22.9014                     196          10.8647               89.1353                  93.5942
         4    85.18

In [10]:
!python extensions/adaptive_routing/context_length_baseline.py \
    --input "{OUTPUT_DIR}/tfidf_analysis.jsonl" \
    --output "{OUTPUT_DIR}/context_length_baseline.csv"

config.json: 100% 661/661 [00:00<00:00, 3.47MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 13.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 147MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 153MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 215MB/s]
Counting context tokens: 100% 19102/19102 [00:05<00:00, 3231.00it/s]
Loaded examples: 19102
Tokenizer/model name: Qwen/Qwen2.5-3B-Instruct

Context-token statistics:
  count: 19102.00
  mean: 210.41
  std: 185.16
  min: 35.00
  10%: 80.00
  20%: 99.00
  30%: 117.00
  40%: 137.00
  50%: 156.00
  60%: 183.00
  70%: 219.00
  80%: 269.00
  90%: 372.00
  max: 1301.00

Always Direct:
  Macro EM: 79.07%
  Macro F1: 85.54%
  Average generated tokens: 5.93
  Total generated tokens: 113257

Always TISER:
  Macro EM: 86.14%
  Macro F1: 90.73%
  Average generated tokens: 272.00
  Total generated tokens: 5195747

Both-correct examples: 14724
TISER-rescue examples: 1804

Threshold sweep (EM/F1 columns are five-dataset macro percen

In [12]:
from huggingface_hub import notebook_login
notebook_login()

api = HfApi()

api.upload_folder(
    folder_path=str(OUTPUT_DIR),
    path_in_repo="extension_1",
    repo_id=HF_REPO,
    repo_type="dataset",
)

CommitInfo(commit_url='https://huggingface.co/datasets/EliElias/TISER-Evaluation-Results/commit/d638a1731891b99e85df9aefd2051f27e8f3fc06', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d638a1731891b99e85df9aefd2051f27e8f3fc06', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/EliElias/TISER-Evaluation-Results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EliElias/TISER-Evaluation-Results'), pr_revision=None, pr_num=None)